# 09 — Custom arrays with `bw_processing`

**Audience:** Advanced users creating deterministic parameter scenarios on top of exported Premise matrices.

**Prerequisites:** The four matrix CSV files produced by Premise, `bw_processing`, `bw2calc`, NumPy, and pandas.

**Learning goals:** build a static matrix datapackage, identify related technosphere and biosphere exchanges, add synchronized scenario arrays, and iterate their scores.


## Outline

1. Load an existing matrix export.
2. Locate a gasoline-car activity and its fuel/CO₂ exchanges.
3. Add aligned technosphere and biosphere arrays.
4. Iterate deterministic scenario columns.


In [ ]:
import csv
import os
from pathlib import Path

import bw2calc as bc
import bw_processing as bwp
import numpy as np
import pandas as pd

matrix_dir_value = os.environ.get("PREMISE_MATRIX_DIR")
if not matrix_dir_value:
    raise RuntimeError("Set PREMISE_MATRIX_DIR to one Premise matrix export directory.")
MATRIX_DIR = Path(matrix_dir_value).expanduser()

required_files = {
    "A_matrix.csv",
    "A_matrix_index.csv",
    "B_matrix.csv",
    "B_matrix_index.csv",
}
missing = sorted(name for name in required_files if not (MATRIX_DIR / name).is_file())
if missing:
    raise FileNotFoundError(f"Missing matrix files: {missing}")


In [ ]:
def read_indices_csv(file_path: Path) -> dict:
    indices = {}
    with file_path.open(encoding="utf-8") as handle:
        rows = csv.reader(handle, delimiter=";")
        next(rows, None)
        for row in rows:
            indices[tuple(str(value) for value in row[:4])] = int(row[4])
    return indices


def read_matrix_csv(file_path: Path):
    array = np.genfromtxt(file_path, delimiter=";", skip_header=1)
    indices = np.array(
        list(zip(array[:, 1].astype(int), array[:, 0].astype(int))),
        dtype=bwp.INDICES_DTYPE,
    )
    return indices, array[:, 2], array[:, -1].astype(bool)


A_indices = read_indices_csv(MATRIX_DIR / "A_matrix_index.csv")
B_indices = read_indices_csv(MATRIX_DIR / "B_matrix_index.csv")
reverse_A = {identifier: label for label, identifier in A_indices.items()}
reverse_B = {identifier: label for label, identifier in B_indices.items()}


## 1. Build the static matrices and method


In [ ]:
static = bwp.create_datapackage()

indices, values, flips = read_matrix_csv(MATRIX_DIR / "A_matrix.csv")
static.add_persistent_vector(
    matrix="technosphere_matrix",
    indices_array=indices,
    data_array=values,
    flip_array=flips,
)

indices, values, _ = read_matrix_csv(MATRIX_DIR / "B_matrix.csv")
static.add_persistent_vector(
    matrix="biosphere_matrix",
    indices_array=indices,
    data_array=values,
    flip_array=None,
)

fossil_co2_ids = [
    identifier
    for label, identifier in B_indices.items()
    if "carbon dioxide, fossil" in label[0].lower()
]
static.add_persistent_vector(
    matrix="characterization_matrix",
    indices_array=np.array(
        [(identifier, identifier) for identifier in fossil_co2_ids],
        dtype=bwp.INDICES_DTYPE,
    ),
    data_array=np.ones(len(fossil_co2_ids)),
)


## 2. Locate the linked exchanges

Build one static LCA first, then map non-zero matrix rows back to exported labels.


In [ ]:
car_id = next(
    identifier
    for label, identifier in A_indices.items()
    if "transport, passenger car" in label[0].lower()
    and "gasoline" in str(label).lower()
)

base_lca = bc.LCA(
    demand={car_id: 1},
    data_objs=[static],
    use_distributions=False,
)
base_lca.lci()
base_lca.lcia()

activity_column = base_lca.dicts.activity[car_id]
technosphere_rows = np.flatnonzero(
    base_lca.technosphere_matrix[:, activity_column].toarray().ravel()
)
supplier_ids = [base_lca.dicts.product.reversed[row] for row in technosphere_rows]
gasoline_id = next(
    identifier
    for identifier in supplier_ids
    if reverse_A.get(identifier, ("",))[0] == "market for petrol, low-sulfur"
)

biosphere_rows = np.flatnonzero(
    base_lca.biosphere_matrix[:, activity_column].toarray().ravel()
)
flow_ids = [base_lca.dicts.biosphere.reversed[row] for row in biosphere_rows]
co2_id = next(
    identifier
    for identifier in flow_ids
    if reverse_B.get(identifier, ("",))[0] == "Carbon dioxide, fossil"
)


## 3. Create synchronized scenario arrays

Every column represents one deterministic scenario. Fuel use and direct CO₂ must have the same number and order of columns.


In [ ]:
gasoline_values = np.arange(0.04, 0.10, 0.01)
co2_values = gasoline_values * 3.15

scenarios = bwp.create_datapackage(sequential=True)
scenarios.add_persistent_array(
    matrix="technosphere_matrix",
    indices_array=np.array([(gasoline_id, car_id)], dtype=bwp.INDICES_DTYPE),
    data_array=gasoline_values.reshape(1, -1),
    flip_array=np.array([True]),
)
scenarios.add_persistent_array(
    matrix="biosphere_matrix",
    indices_array=np.array([(co2_id, car_id)], dtype=bwp.INDICES_DTYPE),
    data_array=co2_values.reshape(1, -1),
)


## 4. Iterate scenario columns


In [ ]:
lca = bc.LCA(
    demand={car_id: 1},
    data_objs=[static, scenarios],
    use_distributions=False,
    use_arrays=True,
)
lca.lci()
lca.lcia()

rows = []
for position, (fuel, direct_co2) in enumerate(zip(gasoline_values, co2_values)):
    if position:
        next(lca)
    rows.append(
        {
            "gasoline input": fuel,
            "direct fossil CO2": direct_co2,
            "score": float(lca.score),
        }
    )

pd.DataFrame(rows)


## Pitfalls and extension

- Keep related arrays aligned and advance them through one sequential datapackage.
- The arrays enumerate scenarios; they are not probability distributions.
- Verify exchange labels instead of assuming matrix IDs are stable across exports.

## Exercise

Replace the linear fuel range with three named efficiency cases and add the case names to the result table.


In [ ]:
exercise_cases = {
    "efficient": 0.04,
    "reference": 0.06,
    "inefficient": 0.08,
}
exercise_co2 = {name: value * 3.15 for name, value in exercise_cases.items()}
